# 第 0 章：环境搭建、学习地图与"什么是强化学习"

> 这一章不教任何算法，目的是让你**直观看到一次完整的 RL 过程**，并确认所有环境就绪。

## 学习目标

读完本章后你应该能：

1. 描述强化学习的 **agent-environment loop**
2. 区分**监督学习 / 强化学习 / RLHF** 三种范式
3. 跑通本课程的第一个环境 `ClickWorld`，看到智能体在交互中"学习"
4. 知道接下来 18 章分别讲什么、以及一条"快速通道"

In [ ]:
# 自动设置 sys.path，让 notebook 能找到根目录下的 rlenvs/ 和 utils/
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 载入 numpy 和 matplotlib
# 在 Jupyter 里 IPython 自动激活 inline 后端，图会嵌在 cell 输出里。
# 不要用 matplotlib.use() 手动切，那会覆盖 IPython 的默认设置导致图不显示。
import numpy as np
import matplotlib.pyplot as plt

print(f"Python: {sys.version.split()[0]}")
print(f"numpy : {np.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"backend : {plt.get_backend()}  (应是 inline / widget 才能显示图)")

## 0.1 环境自检

下面这个 cell 会检查我们后续章节需要的所有依赖。如果某行显示 **❌**，先在终端执行：

```bash
pip install -r requirements.txt
```

然后再回来运行此 cell。

In [ ]:
import importlib
checks = [
    ('numpy',       'numpy'),
    ('matplotlib',  'matplotlib'),
    ('scipy',       'scipy'),
    ('ipywidgets',  'ipywidgets'),
    ('ipympl',      'ipympl'),
    ('tqdm',        'tqdm'),
    ('torch',       'torch（Phase 2 起需要，可选）'),
]

ok = True
for mod, desc in checks:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', '?')
        print(f"  [✓] {desc:<35} {ver}")
    except ImportError:
        print(f"  [✗] {desc:<35} 未安装")
        ok = False

print()
print("全部就绪 ✓" if ok else "存在缺失依赖，请先 pip install")

## 0.1b Jupyter 速成（第一次用 notebook 必读）

如果你没用过 Jupyter notebook，花 3 分钟了解下面几件事，能避免后面 90% 的"跑不通"：

| 操作 | 怎么做 | 说明 |
|---|---|---|
| 运行 cell | 选中后按 `Shift + Enter` | 运行当前 cell 并跳到下一个 |
| 在上方/下方插入 cell | 先按 `Esc`，再按 `A` / `B` | A = above, B = below |
| 删除 cell | 先按 `Esc`，再连按 `D D` | |
| 重启并全部重跑 | 菜单 Kernel → Restart Kernel and Run All Cells | **最常见的修复手段** |

三个关键认知：

1. **cell 必须按顺序运行**：后面的 cell 依赖前面 cell 定义的变量。跳着运行报 `NameError`，先回头重跑前面的 cell。
2. **kernel 的记忆 ≠ 你看到的代码**：改了 cell 但没重跑，执行的还是旧代码。出现诡异结果时，Restart Kernel and Run All Cells。
3. **每章开始前建议 Restart & Run All**：这是验证一章真的能在你机器上跑通的标准方法，本教材每章都控制在 10 分钟以内。

> 📖 **怎么检验自己学会了？** 根目录的 **`STUDY_GUIDE.md`** 给全部 19 章各配了自测题（答案可折叠）。建议每学完一章就去做对应的自测，答不上来再回头复习。

## 0.2 什么是强化学习？

强化学习和监督学习最大的区别可以用一句话概括：

> **监督学习**有标准答案（标签），**强化学习**只有延迟的、稀疏的奖励。

| | 监督学习 | 强化学习 |
|---|---|---|
| 数据 | `(x, y)` 标签对 | `(s, a, r, s')` 转移序列 |
| 反馈 | 即时、确定 | 延迟、稀疏、随机 |
| 目标 | 拟合 `y = f(x)` | 最大化累计奖励 `Σ γ^t r_t` |
| 决策 | 一次推理 | 序列决策、当前动作影响未来 |

举一个最简单的例子：**下围棋**。
- 监督学习的方式：给 AI 一堆 `(棋盘, 高手落子)` 的样本，让它模仿高手。
- 强化学习的方式：让 AI 自己和自己下，赢了 +1 输了 -1。AI 必须**自己发现**"哪几步是好棋"——这就是 **credit assignment 问题**。

### Agent–Environment Loop

```
        action a_t
   ┌─────────────────┐
   │                 ▼
[ Agent ]      [ Environment ]
   ▲                 │
   │                 │
   └─ state s_{t+1}, reward r_{t+1} ─┘
```

每一时刻 $t$：

1. Agent 根据当前状态 $s_t$，按策略 $\pi(a|s)$ 选择动作 $a_t$
2. Environment 接收 $a_t$，转移为新状态 $s_{t+1}$，并给一个标量奖励 $r_{t+1}$
3. Agent 的目标是最大化**期望累计奖励**：

$$
J(\pi) = \mathbb{E}_\pi\left[ \sum_{t=0}^{\infty} \gamma^t r_{t+1} \right]
$$

其中 $\gamma \in [0, 1]$ 是**折扣因子**（Ch02 会详细讲为什么要折扣）。

## 0.3 玩具演示：ClickWorld

现在让我们真的"动手"看一下这个循环。

**`ClickWorld`** 是一个 $10 \times 10$ 的网格：
- 智能体（蓝点）从**左上角** $(0, 0)$ 出发
- 目标在**右下角** $(9, 9)$（金色，奖励 +1）
- 中间有一个陷阱 $(5, 5)$（红色，奖励 -1）
- 抵达目标 +1、踩到陷阱 -1、其他 0

我们对比两种"策略"：

1. **纯随机游走**：每步等概率往上/下/左/右走（撞墙就原地不动）
2. **贪心策略**：每步朝目标方向走

下面的演示是**逐帧动画**。matplotlib 的 jshtml 播放器自带这些控件：

| 按钮 | 功能 |
|---|---|
| ▶ / ⏸ | 播放 / 暂停 |
| − / + | 减慢 / 加快播放速度 |
| ↻ | 循环播放（相当于自动重跑） |
| 进度条 | 拖动到任意帧 |

**想换种子重跑**：改下面 cell 里的 `SEED = 0`（任意整数），然后 Shift+Enter 重跑 cell。
**想调初始速度**：改 `INTERVAL_MS = 200`（毫秒，越大越慢）。

In [ ]:
from rlenvs import ClickWorld
import utils  # 触发中文字体配置（utils/viz.py 顶部设了 Microsoft YaHei 等）
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Circle, Rectangle
from IPython.display import HTML


# ---------- 你可以调的两个参数 ----------
SEED = 0              # 改成任意整数，重跑 cell 看不同结果
INTERVAL_MS = 200     # 帧间隔（毫秒），越大越慢；推荐 100~500
# ------------------------------------

GRID_SIZE = 10
START = (0, 0)
GOAL = (GRID_SIZE - 1, GRID_SIZE - 1)  # (9, 9)，最远的对角
TRAP = (5, 5)


# ---------- 工具函数：跑轨迹 + 生成动画 ----------
def collect_trajectory(env, policy_fn, n_steps, start=START):
    """按 policy_fn 跑 n_steps 步，返回 (states, rewards, reached_goal)。"""
    env.reset()
    env.state = start  # ClickWorld.state 是普通属性，直接赋值即可（默认 reset 是随机起点）
    env.t = 0
    env.reward_history = []
    env.trajectory = [env.state]
    states = [env.state]
    rewards = [0.0]
    reached = False
    for _ in range(n_steps):
        a = policy_fn(env.state)
        _, r, done, _ = env.step(a)
        states.append(env.state)
        rewards.append(r)
        if done:
            reached = True
            for _ in range(3):  # 终止后多停 3 帧让画面定住
                states.append(env.state); rewards.append(0.0)
            break
    return states, rewards, reached


def animate_clickworld(env, states, rewards, interval_ms=200):
    """把 (states, rewards) 渲染成 FuncAnimation。"""
    fig, ax = plt.subplots(figsize=(5, 5))
    size = env.size

    def draw(k):
        ax.clear()
        for i in range(size + 1):
            ax.axhline(i, color='lightgray', linewidth=0.8)
            ax.axvline(i, color='lightgray', linewidth=0.8)
        for (r, c) in env.penalties:
            ax.add_patch(Rectangle((c, r), 1, 1, color='crimson', alpha=0.5))
            ax.text(c + 0.5, r + 0.5, 'X', ha='center', va='center',
                    fontsize=14, fontweight='bold', color='white')
        if env.goal is not None:
            gr, gc = env.goal
            ax.add_patch(Rectangle((gc, gr), 1, 1, color='gold', alpha=0.7))
            ax.text(gc + 0.5, gr + 0.5, 'G', ha='center', va='center',
                    fontsize=14, fontweight='bold')
        # 起点标记
        sr, sc = START
        ax.text(sc + 0.5, sr + 0.5, 'S', ha='center', va='center',
                fontsize=11, color='green', fontweight='bold')
        # 走过的轨迹
        if k > 0:
            ys = [r + 0.5 for r, c in states[:k + 1]]
            xs = [c + 0.5 for r, c in states[:k + 1]]
            ax.plot(xs, ys, '-', color='steelblue', alpha=0.6, linewidth=1.5)
        # 当前位置
        r, c = states[k]
        ax.add_patch(Circle((c + 0.5, r + 0.5), 0.3, color='navy'))
        cum = sum(rewards[:k + 1])
        ax.set_xlim(0, size); ax.set_ylim(0, size)
        ax.set_aspect('equal')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'step {k}   累计奖励 = {cum:.2f}')

    anim = animation.FuncAnimation(
        fig, draw, frames=len(states),
        interval=interval_ms, blit=False, repeat=True,
    )
    plt.close(fig)
    return anim


def random_policy(s):
    return int(np.random.randint(4))


# ---------- 跑一遍随机游走并显示动画 ----------
np.random.seed(SEED)  # 让随机策略也可复现（ClickWorld 内部有自己的 rng，但 random_policy 用全局）
env = ClickWorld(size=GRID_SIZE, seed=SEED)
env.set_goal(GOAL)
env.set_penalty(TRAP)

N_STEPS = 80  # 给随机游走 80 步看能不能到
states, rewards, reached = collect_trajectory(env, random_policy, n_steps=N_STEPS)
status = f"第 {len(states) - 4} 步到达目标" if reached \
         else f"{N_STEPS} 步内未到达目标"
print(f"随机游走 {N_STEPS} 步：{status}")
print(f"累计奖励 = {sum(rewards):.2f}")
anim = animate_clickworld(env, states, rewards, interval_ms=INTERVAL_MS)
HTML(anim.to_jshtml())

看完了随机游走，你应该看到：**纯随机的智能体在网格里乱转**——有时绕回起点、有时撞墙、偶尔接近目标又被带偏。即便 80 步给它够长，也经常到不了对角的目标。

要让智能体"聪明地"走向目标，我们需要给它一个**策略**——这就是接下来 18 章要研究的全部内容。

下面我们用一个简单的"贪心策略"演示：每步朝目标方向（曼哈顿距离最短的方向）走。看看它能不能又快又安全地到达目标。

In [ ]:
# 定义一个朝 goal 走的贪心策略
# 注意 ClickWorld 的动作约定：0=上, 1=下, 2=左, 3=右（和 GridWorld 不一样！）
def greedy_policy(state, goal=GOAL):
    """
    朝 goal 方向走的贪心策略：优先拉近行/列差距更大的那个轴。
    ClickWorld actions: 0=上, 1=下, 2=左, 3=右
    """
    dr = goal[0] - state[0]   # 正：目标在下方
    dc = goal[1] - state[1]   # 正：目标在右方
    if abs(dc) > abs(dr):
        return 3 if dc > 0 else 2   # 朝右 / 朝左
    elif dr != 0:
        return 1 if dr > 0 else 0   # 朝下 / 朝上
    return 0  # 已到目标


# 重新建一个干净的环境（避免上面随机游走污染状态）
env = ClickWorld(size=GRID_SIZE, seed=SEED)
env.set_goal(GOAL)
env.set_penalty(TRAP)

states, rewards, reached = collect_trajectory(env, greedy_policy, n_steps=30)
n_used = len(states) - 4 if reached else len(states) - 1
print(f"贪心策略：{n_used} 步到达目标，累计奖励 = {sum(rewards):.2f}")
anim = animate_clickworld(env, states, rewards, interval_ms=INTERVAL_MS)
HTML(anim.to_jshtml())

看，**有策略**比**没策略**强多了——贪心策略 18 步直达目标。

但是！注意一个细节：贪心策略一路上**踩了陷阱 (5, 5)**，扣了 1 分；最后到达目标 +1 分；**净奖励 = 0**。

如果有个策略能**绕开陷阱同时还能到目标**，它的净奖励会是 +1——比贪心更好。

这就引出了 RL 的核心问题：

> **怎么自动找到一个能拿到尽可能多奖励的策略 $\pi$？**

注意是**"尽可能多奖励"**，不是"能到目标"。也许绕远路避开陷阱比直走更好；也许中间还有更复杂的权衡。**怎么自动找出这种策略**，就是接下来 18 章的全部内容。

## 0.4 学习地图

我们用 4 个 Phase（共 19 章）带你从零基础到能用 PPO / GRPO 训练 LLM、再到研究前沿：

```
Phase 1：经典 RL 基础（你现在在这里）
├── Ch00 环境搭建 + 全景
├── Ch01 多臂老虎机：探索 vs 利用
├── Ch02 MDP + 贝尔曼方程：RL 的数学语言
├── Ch03 动态规划：当你"知道一切"时怎么求解
├── Ch04 TD 学习：从样本中学习
└── Ch05 Q-learning / SARSA：第一个完整的控制算法

Phase 2：策略梯度 + PPO
├── Ch05b PyTorch 速成（没用过 PyTorch 的读者，进 Ch06 前先读）
├── Ch06 DQN + 函数逼近
├── Ch07 策略梯度定理
├── Ch08 Actor-Critic + GAE
└── Ch09 TRPO + PPO：现代 RL 的中流砥柱

Phase 3：LLM RLHF + GRPO（终极目标）
├── Ch10 从零搭 TinyGPT
├── Ch11 Reward Modeling
├── Ch12 RLHF-PPO (InstructGPT 配方)
├── Ch13 GRPO (DeepSeek-R1 的核心)
├── Ch14 DPO / KTO
└── Ch15 终局项目

Phase 4：研究前沿
├── Ch16 PRM（过程奖励模型）
├── Ch17 Self-Play + Constitutional AI / RLAIF
└── Ch18 Offline RL（CQL / IQL / Decision Transformer）
```

### 🏁 Fast-track 路径（如果你赶时间）

如果你已经有 RL 基础、想尽快到 LLM RLHF 部分：

**Ch00 → Ch01 → Ch05 → Ch07 → Ch09 → Ch13**（约 20 小时直达 GRPO）

> 快速通过 Ch06-09 需要会 PyTorch——没写过的话把 **Ch05b** 插进 fast-track 里，多花一小时值得。

## 0.5 怎么使用这套教材

每个 notebook 都遵循这个结构：

1. **学习目标**：开头 3-5 条 bullet
2. **概念 + 数学推导**：LaTeX 公式，关键证明放在可折叠 `<details>` 块里
3. **数值验证**：关键公式后面有代码 cell，用数值方法验证
4. **交互式 widget**：滑块调参，实时看效果
5. **从零实现**：你自己写代码，我们提供脚手架
6. **可视化**：训练曲线、动画
7. **练习 + 自测**：练习的参考答案在 `solutions/` 目录；每章的自测题集中在根目录 `STUDY_GUIDE.md`

### 一些小贴士

- 每章都能**独立跑通**（10 分钟以内），不会卡死
- 重要概念会反复出现（比如 on-policy vs off-policy），第一次见是引子，第二次见是深入
- 公式不会的，跳过！下一章还会再讲
- **画图/动画显示说明**：本教材的动画用 `to_jshtml()` 生成 HTML 播放器、滑块用 ipywidgets 控件——普通 inline 后端就能用，**不需要** `%matplotlib widget`。如果图不显示，Restart Kernel and Run All Cells 即可

## 0.6 一些核心术语速查

| 术语 | 英文 | 一句话解释 |
|---|---|---|
| 状态 | state $s$ | 环境的当前快照 |
| 动作 | action $a$ | agent 能选的操作 |
| 奖励 | reward $r$ | 环境给 agent 的即时反馈 |
| 回报 | return $G$ | 从某时刻起累计的折扣奖励 $\sum \gamma^t r$ |
| 策略 | policy $\pi$ | state → action 的映射 |
| 价值 | value $V^\pi(s)$ | 在状态 $s$ 下、用策略 $\pi$ 的期望回报 |
| 动作价值 | $Q^\pi(s,a)$ | 在 $s$ 选 $a$、之后用 $\pi$ 的期望回报 |
| 折扣因子 | discount $\gamma$ | 未来奖励的衰减系数，常 0.9~0.99 |
| Episode | / | 一次完整的轨迹（从开始到结束） |
| On-policy | / | 用当前策略采样的数据训练当前策略 |
| Off-policy | / | 可以用旧策略采的数据训练新策略 |

## 0.7 小结

- ✅ 环境已就绪
- ✅ 理解了 agent-environment loop
- ✅ 见识了 `ClickWorld`：策略决定一切
- ✅ 知道接下来 18 章学什么

下一章：**第 1 章 — 多臂老虎机**。我们将用最简单的 RL 问题，引入两个核心矛盾：**探索 vs 利用** 和 **信用分配**。

> 💡 提示：如果上面的 `ClickWorld` 动画没有显示，多半是 cell 没按顺序运行——Restart Kernel and Run All Cells 一次即可。动画是 HTML 播放器（jshtml），不需要 `%matplotlib widget`。